In [2]:
# ========================================================================
# ERICSSON IMAGE RAG DEMONSTRATION
# ========================================================================
# Purpose: Semantic search across equipment images using AI-generated descriptions
#
# Architecture:
# 1. GPT-4o Vision: Converts images → text descriptions
# 2. Ada-002 Embeddings: Converts descriptions → 1536-D vectors
# 3. ChromaDB: Stores vectors for similarity search
# 4. Search: Query → vector → find similar equipment
#
# Use Case: Field technicians can search equipment by natural language
# Example: "equipment with ethernet ports" finds relevant images
# ========================================================================

print("🚀 INITIALIZING IMAGE RAG SYSTEM")
print("=" * 70)

# ========================================================================
# STEP 1: Import Required Libraries
# ========================================================================
# What: Core Python libraries for API calls and data processing
# Why: We use direct HTTP requests (no OpenAI SDK) to avoid version conflicts
# ========================================================================

import requests      # For making HTTP calls to Azure OpenAI API
import base64        # For encoding images to base64 (required by GPT-4o Vision)
import chromadb      # Vector database for semantic search
import os            # For file system operations (finding image files)
import json          # For handling JSON responses from API

print("✅ Libraries imported")

# ========================================================================
# STEP 2: Configure Azure OpenAI Connection
# ========================================================================
# What: Your Azure OpenAI credentials and endpoint
# Why: Authenticates access to GPT-4o (vision) and Ada-002 (embeddings)
#
# Resource: najr-miyonro1-eastus2 (East US 2)
# Deployments: gpt-4o, text-embedding-ada-002
# ========================================================================

API_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"

print("✅ Azure OpenAI configured (East US 2)")

# ========================================================================
# STEP 3: Define Image Description Function
# ========================================================================
# What: Uses GPT-4o Vision to analyze equipment images
# Why: Converts visual information to searchable text
#
# Process:
# 1. Read image file from disk
# 2. Encode to base64 (API requirement for image data)
# 3. Send to GPT-4o with vision prompt
# 4. Receive detailed text description
#
# Input: Image file path (e.g., "radio-dot-8413-high.jpg")
# Output: Text description (e.g., "This is an Ericsson Radio Dot 8413...")
# ========================================================================

def describe_image(image_path):
    """
    Generate detailed description of equipment image using GPT-4o Vision
    
    Args:
        image_path (str): Path to image file (.jpg, .png)
    
    Returns:
        str: Detailed description of equipment, ports, labels, specs
    """
    # Read image file and convert to base64
    # Why base64: API requires image data in base64 format
    with open(image_path, "rb") as f:
        base64_image = base64.b64encode(f.read()).decode('utf-8')
    
    # Construct API endpoint URL
    # Deployment: gpt-4o (vision-capable model)
    # API version: 2024-02-01 (stable version)
    url = f"{ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01"
    
    # Set headers for authentication and content type
    headers = {
        "api-key": API_KEY,              # Azure authentication
        "Content-Type": "application/json"  # JSON payload
    }
    
    # Construct request payload
    # Message format: text prompt + base64 image
    data = {
        "messages": [{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this Ericsson telecom equipment in detail. Include all visible ports, connectors, labels, model numbers, and technical specifications."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        }],
        "max_tokens": 200  # Limit response length (cost optimization)
    }
    
    # Make API call
    response = requests.post(url, headers=headers, json=data)
    
    # Extract description from response
    return response.json()["choices"][0]["message"]["content"]

print("✅ Image description function defined (GPT-4o Vision)")

# ========================================================================
# STEP 4: Define Embedding Function
# ========================================================================
# What: Converts text to 1536-dimensional vector using Ada-002
# Why: Vectors enable semantic similarity search (finding related content)
#
# How it works:
# - "radio equipment" → [0.023, -0.145, 0.891, ... 1536 numbers]
# - "ethernet port" → [0.012, -0.098, 0.234, ... 1536 numbers]
# - Similar meanings = similar vectors (mathematically close in space)
#
# Input: Text string
# Output: List of 1536 floating-point numbers (embedding vector)
# ========================================================================

def get_embedding(text):
    """
    Convert text to embedding vector for semantic search
    
    Args:
        text (str): Text to embed (description or query)
    
    Returns:
        list: 1536-dimensional embedding vector
    """
    # Construct API endpoint URL
    # Deployment: text-embedding-ada-002
    url = f"{ENDPOINT}openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01"
    
    # Set headers
    headers = {
        "api-key": API_KEY,
        "Content-Type": "application/json"
    }
    
    # Construct payload
    data = {"input": text}
    
    # Make API call
    response = requests.post(url, headers=headers, json=data)
    
    # Extract embedding vector from response
    return response.json()["data"][0]["embedding"]

print("✅ Embedding function defined (Ada-002)")

# ========================================================================
# STEP 5: Initialize Vector Database (ChromaDB)
# ========================================================================
# What: In-memory vector database for storing and searching embeddings
# Why: Enables fast semantic similarity search across image descriptions
#
# How it works:
# - Store: [description vector, original text, metadata]
# - Search: Convert query → vector → find nearest neighbor vectors
# - Return: Original descriptions + metadata of similar items
#
# Note: In-memory means data clears on restart (fast for demos)
# ========================================================================

chroma_client = chromadb.Client()  # Create ChromaDB client

# Delete old collection if exists (ensures fresh start)
try:
    chroma_client.delete_collection("ericsson_equipment")
except:
    pass  # Collection doesn't exist yet, that's fine

# Create new collection for equipment images
# Collection: Like a table in a database, holds related vectors
collection = chroma_client.create_collection("ericsson_equipment")

print("✅ ChromaDB initialized (in-memory vector database)")

# ========================================================================
# STEP 6: Process All Equipment Images
# ========================================================================
# What: Load all .jpg images, generate descriptions, store in database
# Why: Makes all equipment searchable by natural language queries
#
# Process for each image:
# 1. GPT-4o Vision: Image → Text description
# 2. Ada-002: Description → 1536-D vector
# 3. ChromaDB: Store vector + description + metadata
#
# Result: All images indexed and searchable
# ========================================================================

print("\n" + "=" * 70)
print("PROCESSING EQUIPMENT IMAGES")
print("=" * 70)

# Find all .jpg image files in current directory
# Why .jpg: Your equipment images are in JPG format
image_files = [f for f in os.listdir('.') if f.lower().endswith('.jpg')]

print(f"📁 Found {len(image_files)} images in directory\n")

# Process each image
for index, image_filename in enumerate(image_files):
    print(f"[{index + 1}/{len(image_files)}] Processing: {image_filename}")
    
    # Step 1: Generate description with GPT-4o Vision
    # This is the "AI understanding" step - converts pixels to meaning
    description = describe_image(image_filename)
    print(f"   📝 Description generated: {description[:80]}...")
    
    # Step 2: Convert description to embedding vector
    # This is the "vectorization" step - converts text to numbers
    embedding_vector = get_embedding(description)
    print(f"   🔢 Embedding created: 1536-dimensional vector")
    
    # Step 3: Store in ChromaDB
    # Stores: vector (for search) + description (for display) + metadata (filename)
    collection.add(
        embeddings=[embedding_vector],           # The 1536-D vector
        documents=[description],                 # Original text description
        metadatas=[{"filename": image_filename}], # Image filename for reference
        ids=[f"image_{index}"]                   # Unique identifier
    )
    print(f"   ✅ Stored in database\n")

print("=" * 70)
print(f"✅ SUCCESS: {len(image_files)} images indexed and searchable")
print("=" * 70)

# ========================================================================
# STEP 7: Define Search Function
# ========================================================================
# What: Semantic search across equipment images
# Why: Allows finding equipment by features, not just keywords
#
# How it works:
# 1. User query: "equipment with ethernet ports"
# 2. Convert query to vector: [0.123, -0.456, ...]
# 3. Find closest vectors in database (cosine similarity)
# 4. Return: Top N matching descriptions + image filenames
#
# Example:
# - Query: "rooftop equipment" → Finds AIR_6428_Rooftop image
# - Query: "radio dot" → Finds radio-dot-8413-high image
# ========================================================================

def search_equipment(query, num_results=3):
    """
    Search for equipment images using natural language query
    
    Args:
        query (str): Natural language search query
                    Examples: "radio equipment", "ethernet ports", "outdoor gear"
        num_results (int): Number of results to return (default: 3)
    
    Returns:
        None (prints results to console)
    
    Example:
        search_equipment("equipment with network ports")
        → Returns images with visible RJ45/ethernet ports
    """
    print(f"\n🔍 SEARCH QUERY: '{query}'")
    print("=" * 70)
    
    # Step 1: Convert query to embedding vector
    # Why: Need to compare query against stored description vectors
    query_embedding = get_embedding(query)
    
    # Step 2: Search ChromaDB for similar vectors
    # ChromaDB uses cosine similarity to find closest matches
    results = collection.query(
        query_embeddings=[query_embedding],  # Your query vector
        n_results=num_results                # How many results to return
    )
    
    # Step 3: Display results
    # Results contain: documents (descriptions) and metadatas (filenames)
    for i, (description, metadata) in enumerate(zip(
        results['documents'][0],   # List of matching descriptions
        results['metadatas'][0]    # List of corresponding metadata
    )):
        print(f"\n📷 RESULT {i + 1}:")
        print(f"   Image: {metadata['filename']}")
        print(f"   Description: {description[:100]}...")
    
    print("\n" + "=" * 70)

print("\n✅ Search function defined")

# ========================================================================
# STEP 8: DEMONSTRATION - Run Sample Searches
# ========================================================================
# What: Example searches showing Image RAG capabilities
# Why: Demonstrates semantic search vs keyword matching
#
# Key Point: These searches work by MEANING, not keywords
# - "radio equipment" finds images even if description doesn't say "radio"
# - "ethernet ports" finds images with RJ45 connectors
# - "rooftop installation" finds outdoor-mountable equipment
# ========================================================================

print("\n" + "=" * 70)
print("DEMONSTRATION: SEMANTIC IMAGE SEARCH")
print("=" * 70)

# Demo Query 1: Equipment Type
print("\n--- DEMO 1: Search by Equipment Type ---")
search_equipment("radio equipment", num_results=3)

# Demo Query 2: Technical Features
print("\n--- DEMO 2: Search by Features ---")
search_equipment("equipment with ethernet or network ports", num_results=2)

# Demo Query 3: Installation Location
print("\n--- DEMO 3: Search by Deployment Type ---")
search_equipment("rooftop or outdoor telecom equipment", num_results=2)

# Demo Query 4: Model Number
print("\n--- DEMO 4: Search by Model ---")
search_equipment("AIR 6428", num_results=1)

# ========================================================================
# DEMO COMPLETE
# ========================================================================

print("\n" + "=" * 70)
print("✅ IMAGE RAG DEMO COMPLETE")
print("=" * 70)
print(f"\n📊 SUMMARY:")
print(f"   • Total images indexed: {len(image_files)}")
print(f"   • Technology: GPT-4o Vision + Ada-002 Embeddings + ChromaDB")
print(f"   • Search type: Semantic (meaning-based, not keyword)")
print(f"\n💡 USE CASE FOR ERICSSON:")
print(f"   • Field technicians search equipment by description")
print(f"   • Find equipment by features (ports, connectors, mounting)")
print(f"   • No manual tagging required - AI generates descriptions")
print(f"   • Scales to thousands of equipment images")
print("\n" + "=" * 70)

🚀 INITIALIZING IMAGE RAG SYSTEM
✅ Libraries imported
✅ Azure OpenAI configured (East US 2)
✅ Image description function defined (GPT-4o Vision)
✅ Embedding function defined (Ada-002)
✅ ChromaDB initialized (in-memory vector database)

PROCESSING EQUIPMENT IMAGES
📁 Found 5 images in directory

[1/5] Processing: AIR_6428_Rooftop_16bit_sRGB_148412 (2).jpg
   📝 Description generated: The image appears to show Ericsson telecommunications equipment installed on a r...
   🔢 Embedding created: 1536-dimensional vector
   ✅ Stored in database

[2/5] Processing: ericsson-radio-system-family-mounting-alternatives-59625-low.jpg
   📝 Description generated: This image features Ericsson telecom equipment, specifically designed for flexib...
   🔢 Embedding created: 1536-dimensional vector
   ✅ Stored in database

[3/5] Processing: radio-dot-8413-high.jpg
   📝 Description generated: This is an example of Ericsson telecom equipment, specifically designed for indo...
   🔢 Embedding created: 1536-dimension

In [ ]:
# ========================================================================
# Cell 1: Install Required Packages for Image RAG
# ========================================================================
# Purpose: Install Python libraries needed for multimodal RAG with images
# 
# Why each package:
# - openai: Azure OpenAI SDK to call GPT-4o (vision) and ada-002 (embeddings)
# - chromadb: Vector database to store image descriptions (same as text RAG)
# - tiktoken: Token counter for OpenAI models (manages context limits)
# - pillow: Python Imaging Library - handles image file operations (NEW for images)
#
# The --quiet flag suppresses detailed installation logs for cleaner output
# ========================================================================

!pip install openai chromadb tiktoken pillow --quiet

print("✅ Packages installed successfully!")

In [ ]:
# ========================================================================
# Cell 2: Import Required Libraries
# ========================================================================
# Purpose: Import all Python libraries we just installed
#
# Why each import:
# - os: Access environment variables (for Azure credentials stored securely)
# - AzureOpenAI: Client to communicate with Azure OpenAI service
# - chromadb: Initialize vector database for storing embeddings
# - embedding_functions: Pre-built functions to generate embeddings
# - base64: Encode images to send to GPT-4o (vision models need base64 format)
# - Image from PIL: Load and manipulate image files before processing
#
# Note: We're adding base64 and PIL (Pillow) which weren't in the text RAG demo
# ========================================================================

import os
from openai import AzureOpenAI
import chromadb
from chromadb.utils import embedding_functions
import base64
from PIL import Image

print("✅ Libraries imported successfully!")

In [ ]:
# ========================================================================
# Cell 3: Configure Azure OpenAI Credentials
# ========================================================================
# Purpose: Set up connection to Azure OpenAI service using your credentials
#
# Why we need this:
# - We'll use GPT-4o for TWO tasks:
#   1. Vision (describing images) - NEW for image RAG
#   2. Chat (generating final answers) - same as text RAG
# - We'll use text-embedding-ada-002 for embeddings (same as your text RAG demo)
#
# Security Note: Credentials are stored as environment variables, not hardcoded
# ========================================================================

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.getenv("BVLfyRpeNZmbaAf4gd44Q9PHHgUL8Sg9UcZE6rTa6ezQM78iICzyJQQJ99BLACYeBjFXJ3w3AAABACOG8jVL"),      # Your Azure OpenAI key
    api_version="2024-02-01",                        # API version
    azure_endpoint=os.getenv("https://rag-demo-openai-najib.openai.azure.com/") # Your Azure endpoint URL
)

# Model Configuration - Define which models we'll use
VISION_MODEL = "gpt-4o"                    # For describing images (vision capability)
EMBEDDING_MODEL = "text-embedding-ada-002"  # For creating vectors (same as text RAG)
CHAT_MODEL = "gpt-4o"                      # For generating final answers

print("✅ Azure OpenAI configured successfully!")
print(f"📊 Models configured:")
print(f"   - Vision: {VISION_MODEL}")
print(f"   - Embeddings: {EMBEDDING_MODEL}")
print(f"   - Chat: {CHAT_MODEL}")

In [ ]:
# ========================================================================
# Cell 3: Configure Azure OpenAI Credentials
# ========================================================================
# Purpose: Set up connection to Azure OpenAI service using your credentials
#
# UPDATED: Now using the East US 2 resource which has GPT-4o deployed
# ========================================================================

# Set environment variables with your Azure OpenAI credentials
os.environ["AZURE_OPENAI_API_KEY"] = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),      
    api_version="2024-02-01",                        
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT") 
)

# Model Configuration - Define which models we'll use
VISION_MODEL = "gpt-4o"                    # For describing images (vision capability)
EMBEDDING_MODEL = "text-embedding-ada-002"  # For creating vectors (same as text RAG)
CHAT_MODEL = "gpt-4o"                      # For generating final answers

print("✅ Azure OpenAI configured successfully!")
print(f"📊 Models configured:")
print(f"   - Vision: {VISION_MODEL}")
print(f"   - Embeddings: {EMBEDDING_MODEL}")
print(f"   - Chat: {CHAT_MODEL}")

In [ ]:
# ========================================================================
# Cell 4: Image Description Function (The "Bridge")
# ========================================================================
# Purpose: Convert images into detailed text descriptions using GPT-4o vision
#
# This is the KEY difference from text RAG:
# - Text RAG: directly chunks and embeds text
# - Image RAG: first describe image → then embed the description
#
# Why GPT-4o?
# - It has vision capabilities - can "see" and understand images
# - Generates technical descriptions perfect for Ericsson equipment
# - Identifies ports, labels, model numbers automatically
#
# Process Flow:
# 1. Load image file from disk
# 2. Encode to base64 (required format for GPT-4o API)
# 3. Send to GPT-4o with custom prompt
# 4. Return detailed text description
# ========================================================================

def describe_image(image_path, prompt="Describe this technical equipment in detail. Identify all ports, labels, connectors, model numbers, and any visible text."):
    """
    Uses GPT-4o vision to generate a detailed text description of an image.
    
    Parameters:
    - image_path: Full path to the image file (e.g., '/path/to/radio.jpg')
    - prompt: Instruction for how to describe the image (customizable)
    
    Returns:
    - description: Detailed text description of the image
    
    Example:
    description = describe_image('/images/radio_unit.jpg')
    """
    
    # Step 1: Read the image file and encode it to base64
    # Why base64? GPT-4o API requires images in base64 string format
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    # Step 2: Call GPT-4o with vision capability
    # We send both the prompt (text) and the image (base64)
    response = client.chat.completions.create(
        model=VISION_MODEL,  # gpt-4o has vision capabilities
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},  # Our instruction
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"  # The image
                        }
                    }
                ]
            }
        ],
        max_tokens=500  # Limit description length
    )
    
    # Step 3: Extract the description from the API response
    description = response.choices[0].message.content
    
    return description

print("✅ Image description function created successfully!")
print("📷 Ready to convert images into searchable text descriptions")

In [ ]:
# ========================================================================
# Cell 5: Test Image Description with GPT-4o Vision
# ========================================================================
# Purpose: Generate a detailed description of the radio equipment image
#
# This demonstrates the "bridge" concept:
# - Input: Image file (radio-dot-8413-high.jpg)
# - Process: GPT-4o analyzes the image
# - Output: Detailed text description
#
# This text description will later be embedded and stored in ChromaDB
# ========================================================================

# Define the path to your uploaded image
# The image is in: /mnt/batch/tasks/shared/LS_root/mounts/clusters/rag-demo-compute-najib/code/Users/naj_r/
image_path = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/rag-demo-compute-najib/code/Users/naj_r/radio-dot-8413-high.jpg"

print("📷 Analyzing image with GPT-4o vision...")
print("=" * 70)

# Call our describe_image function
image_description = describe_image(image_path)

print("\n✅ Image Description Generated:")
print("=" * 70)
print(image_description)
print("=" * 70)

# Show some metadata
print(f"\n📊 Description length: {len(image_description)} characters")
print(f"💾 This description will be embedded and stored in ChromaDB")

In [ ]:
# ========================================================================
# Cell 5: Find the Correct Image Path & Test Description
# ========================================================================
# Purpose: First locate the image file, then generate description
#
# Azure ML stores files in specific locations - we need to find the exact path
# ========================================================================

import os

# Step 1: Find the current working directory
current_dir = os.getcwd()
print(f"📁 Current working directory: {current_dir}")

# Step 2: List all files in the current directory to find our image
print("\n📂 Files in current directory:")
files = os.listdir(current_dir)
for file in files:
    if file.endswith(('.jpg', '.png', '.jpeg')):
        print(f"   🖼️  {file}")

# Step 3: Build the correct path
image_filename = "radio-dot-8413-high.jpg"
image_path = os.path.join(current_dir, image_filename)

print(f"\n✅ Using image path: {image_path}")
print(f"📊 File exists: {os.path.exists(image_path)}")

# Step 4: If file exists, generate description
if os.path.exists(image_path):
    print("\n📷 Analyzing image with GPT-4o vision...")
    print("=" * 70)
    
    image_description = describe_image(image_path)
    
    print("\n✅ Image Description Generated:")
    print("=" * 70)
    print(image_description)
    print("=" * 70)
    
    print(f"\n📊 Description length: {len(image_description)} characters")
    print(f"💾 This description will be embedded and stored in ChromaDB")
else:
    print(f"\n❌ Error: Image file not found at {image_path}")
    print("💡 Please check the file location in the sidebar")

In [ ]:
# ========================================================================
# Cell 5: Search for Image Files Recursively
# ========================================================================
# Purpose: Find where the images are actually stored
# ========================================================================

import os

# Get current directory
current_dir = os.getcwd()
print(f"📁 Current directory: {current_dir}\n")

# Search for the image file recursively
print("🔍 Searching for image files...\n")

found_images = []
for root, dirs, files in os.walk(current_dir):
    for file in files:
        if file.endswith(('.jpg', '.png', '.jpeg')):
            full_path = os.path.join(root, file)
            found_images.append(full_path)
            print(f"🖼️  Found: {file}")
            print(f"   Path: {full_path}\n")

if found_images:
    # Use the first image we found (radio-dot-8413-high.jpg)
    image_path = [img for img in found_images if 'radio-dot-8413' in img][0]
    
    print(f"✅ Selected image: {image_path}")
    print("\n📷 Analyzing image with GPT-4o vision...")
    print("=" * 70)
    
    image_description = describe_image(image_path)
    
    print("\n✅ Image Description Generated:")
    print("=" * 70)
    print(image_description)
    print("=" * 70)
    
    print(f"\n📊 Description length: {len(image_description)} characters")
else:
    print("❌ No image files found!")
    print("💡 You may need to upload the image to the notebook directory")

In [ ]:
# ========================================================================
# Cell 5: Use Image from Same Directory (naj_r folder)
# ========================================================================

import os

# The image is in the same folder as the notebook
current_dir = os.getcwd()
image_path = os.path.join(current_dir, "radio-dot-8413-high.jpg")

print(f"📁 Current directory: {current_dir}")
print(f"🖼️  Image path: {image_path}")
print(f"✅ File exists: {os.path.exists(image_path)}\n")

if os.path.exists(image_path):
    print("📷 Analyzing image with GPT-4o vision...")
    print("=" * 70)
    
    image_description = describe_image(image_path)
    
    print("\n✅ Image Description Generated:")
    print("=" * 70)
    print(image_description)
    print("=" * 70)
    
    print(f"\n📊 Description length: {len(image_description)} characters")
    print(f"💾 This description will be embedded and stored in ChromaDB")
else:
    print("❌ File not found!")
    # List what IS in the directory
    print("\n📂 Files in current directory:")
    for f in os.listdir(current_dir):
        print(f"   - {f}")

In [ ]:
# ========================================================================
# Check Available Models in Your Azure OpenAI
# ========================================================================

# Let's see what you're using in your text RAG demo
print("🔍 Checking your Azure OpenAI deployments...\n")

# Try different common deployment names
test_models = [
    "gpt-4o",
    "gpt-4",
    "gpt-4-vision",
    "gpt-35-turbo",
    "gpt-4-turbo"
]

print("Testing which models are available:\n")
for model in test_models:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "test"}],
            max_tokens=5
        )
        print(f"✅ {model} - AVAILABLE")
    except Exception as e:
        if "DeploymentNotFound" in str(e):
            print(f"❌ {model} - NOT DEPLOYED")
        else:
            print(f"⚠️  {model} - Error: {str(e)[:50]}")

In [ ]:
# What models are you using?
print("Chat Model:", CHAT_MODEL if 'CHAT_MODEL' in dir() else "Not defined")
print("Embedding Model:", EMBEDDING_MODEL if 'EMBEDDING_MODEL' in dir() else "Not defined")

In [ ]:
# Test if the API actually works
try:
    response = client.chat.completions.create(
        model=CHAT_MODEL,  # Using the variable you defined
        messages=[{"role": "user", "content": "Say 'working'"}],
        max_tokens=5
    )
    print(f"✅ SUCCESS! Model '{CHAT_MODEL}' is working!")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ ERROR: {e}")

In [ ]:
# ========================================================================
# Cell 6: Initialize ChromaDB for Image Descriptions
# ========================================================================
# Purpose: Set up a vector database to store image descriptions as embeddings
#
# Why ChromaDB?
# - Same vector database you used in your text RAG demo
# - Stores embeddings (vectors) for semantic search
# - Allows us to search images by text queries
#
# How it works:
# 1. Create a ChromaDB client (in-memory for this demo)
# 2. Set up Azure OpenAI embedding function (ada-002)
# 3. Create a collection called "ericsson_image_rag"
# ========================================================================

# Step 1: Initialize ChromaDB client
chroma_client = chromadb.Client()

# Step 2: Configure Azure OpenAI embedding function
# This is the SAME embedding model from your text RAG demo
azure_openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_base=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_type="azure",
    api_version="2024-02-01",
    model_name=EMBEDDING_MODEL  # text-embedding-ada-002
)

# Step 3: Create a collection for image descriptions
# Note: If collection already exists, this will fail - that's okay for demo purposes
try:
    collection = chroma_client.create_collection(
        name="ericsson_image_rag",
        embedding_function=azure_openai_ef,
        metadata={"description": "Image RAG - Equipment images with GPT-4o descriptions"}
    )
    print(f"✅ Collection created: {collection.name}")
except Exception as e:
    # If collection exists, get it instead
    collection = chroma_client.get_collection(
        name="ericsson_image_rag",
        embedding_function=azure_openai_ef
    )
    print(f"✅ Collection already exists: {collection.name}")

print(f"📊 Total documents in collection: {collection.count()}")

In [ ]:
# ========================================================================
# Cell 6: Initialize ChromaDB for Image Descriptions
# ========================================================================
# Purpose: Set up a vector database to store image descriptions as embeddings
#
# IMPORTANT FIX: Azure OpenAI requires deployment_id (deployment name)
# ========================================================================

# Step 1: Initialize ChromaDB client
chroma_client = chromadb.Client()

# Step 2: Configure Azure OpenAI embedding function
# For Azure OpenAI, we must specify BOTH model_name AND deployment_id
azure_openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_base=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_type="azure",
    api_version="2024-02-01",
    model_name="text-embedding-ada-002",       # Model name
    deployment_id="text-embedding-ada-002"     # Deployment name (REQUIRED for Azure)
)

# Step 3: Create a collection for image descriptions
try:
    collection = chroma_client.create_collection(
        name="ericsson_image_rag",
        embedding_function=azure_openai_ef,
        metadata={"description": "Image RAG - Equipment images with GPT-4o descriptions"}
    )
    print(f"✅ Collection created: {collection.name}")
except Exception as e:
    # If collection exists, get it instead
    collection = chroma_client.get_collection(
        name="ericsson_image_rag",
        embedding_function=azure_openai_ef
    )
    print(f"✅ Collection already exists: {collection.name}")

print(f"📊 Total documents in collection: {collection.count()}")

In [ ]:
# ========================================================================
# Cell 7: Store Image Description as Vector Embedding
# ========================================================================
# Purpose: Add the GPT-4o generated description to ChromaDB
#
# What happens here:
# 1. Take the image description from Cell 5 (stored in image_description variable)
# 2. ChromaDB automatically converts it to a 1536-dimensional vector using ada-002
# 3. Store the vector + original text + metadata in the database
#
# Metadata includes:
# - image_path: so we can show the user which image matched their query
# - equipment_type: for filtering/categorization
# - model: specific model identifier
# ========================================================================

# The image path from Cell 5 (update this if you used a different image)
image_filename = "radio-dot-8413-high.jpg"
image_path_for_metadata = image_filename  # We'll store just the filename

# Add the description to ChromaDB
# The description will be automatically embedded by ada-002
collection.add(
    documents=[image_description],  # The GPT-4o description from Cell 5
    metadatas=[{
        "image_filename": image_filename,
        "equipment_type": "Radio Dot",
        "model": "8413",
        "description_method": "GPT-4o Vision"
    }],
    ids=["radio_dot_8413_001"]  # Unique identifier for this image
)

print("✅ Image description stored in ChromaDB!")
print(f"📊 Total documents in collection: {collection.count()}")
print(f"\n📝 Stored description preview:")
print(f"{image_description[:200]}...")  # Show first 200 characters

In [ ]:
# ========================================================================
# Cell 8: Image RAG Query Function
# ========================================================================
# Purpose: Search for images using natural language queries
#
# How it works:
# 1. User asks a question (e.g., "What equipment has an Ethernet port?")
# 2. Question is converted to embedding vector (ada-002)
# 3. ChromaDB finds descriptions with similar vectors (semantic search)
# 4. Return the matching image descriptions + metadata
#
# This is the core of Image RAG - searching images by their AI-generated descriptions
# ========================================================================

def search_images(query, n_results=2):
    """
    Search for images using natural language queries
    
    Parameters:
    - query: Text query (e.g., "radio equipment with ports")
    - n_results: Number of results to return (default: 2)
    
    Returns:
    - Dictionary with search results and metadata
    """
    
    # Step 1: Search the vector database
    # ChromaDB automatically converts the query to a vector and finds similar descriptions
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    # Step 2: Extract results
    retrieved_docs = results['documents'][0]  # The descriptions
    retrieved_metadata = results['metadatas'][0]  # The metadata (image paths, etc.)
    retrieved_distances = results['distances'][0]  # Similarity scores (lower = better match)
    
    # Step 3: Display results
    print(f"🔍 Search Query: '{query}'")
    print("=" * 70)
    
    for i, (doc, meta, distance) in enumerate(zip(retrieved_docs, retrieved_metadata, retrieved_distances)):
        print(f"\n📷 Result {i+1}:")
        print(f"   Image: {meta.get('image_filename', 'Unknown')}")
        print(f"   Equipment: {meta.get('equipment_type', 'Unknown')}")
        print(f"   Model: {meta.get('model', 'Unknown')}")
        print(f"   Similarity Score: {1 - distance:.3f} (higher = better match)")
        print(f"   Description Preview: {doc[:150]}...")
    
    print("=" * 70)
    
    return {
        "query": query,
        "results": retrieved_docs,
        "metadata": retrieved_metadata,
        "scores": [1 - d for d in retrieved_distances]  # Convert distance to similarity
    }

print("✅ Image search function created successfully!")
print("📷 Ready to search images by description")

In [ ]:
# ========================================================================
# Cell 9: Test Image Search (Image RAG Demo)
# ========================================================================
# Purpose: Demonstrate Image RAG by searching for the radio equipment
#
# We'll test several queries to show how semantic search works
# ========================================================================

print("🚀 Testing Image RAG - Searching for Equipment\n")

# Test Query 1: Direct search for radio equipment
print("\n" + "="*70)
print("TEST 1: Looking for radio equipment")
print("="*70)
result1 = search_images("What radio equipment do we have?")

# Test Query 2: Search by specific features
print("\n" + "="*70)
print("TEST 2: Looking for equipment with Ethernet port")
print("="*70)
result2 = search_images("Show me equipment with an Ethernet port")

# Test Query 3: Search by technical specs
print("\n" + "="*70)
print("TEST 3: Looking for PoE powered devices")
print("="*70)
result3 = search_images("What devices support Power over Ethernet?")

# Test Query 4: Search by model number
print("\n" + "="*70)
print("TEST 4: Looking for specific model")
print("="*70)
result4 = search_images("RD 222.84 equipment")

print("\n✅ Image RAG Demo Complete!")
print("💡 Notice how the system finds the radio image based on semantic meaning,")
print("   not just keyword matching!")

In [ ]:
# ========================================================================
# Cell 10: Batch Process All Images in naj_r Folder
# ========================================================================
# Purpose: Describe and store ALL Ericsson equipment images
#
# Process:
# 1. Find all image files in current directory
# 2. For each image:
#    - Generate description with GPT-4o
#    - Store in ChromaDB with metadata
# 3. Skip images already processed
# ========================================================================

import os
import time

# Step 1: Find all image files
current_dir = os.getcwd()
image_files = [f for f in os.listdir(current_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"📁 Found {len(image_files)} images in directory")
print(f"📊 Current documents in collection: {collection.count()}\n")

# Step 2: Process each image
processed_count = 0
skipped_count = 0

for idx, image_file in enumerate(image_files, 1):
    image_id = f"ericsson_img_{idx:03d}"
    
    # Check if already processed (by checking if ID exists)
    try:
        existing = collection.get(ids=[image_id])
        if existing['ids']:
            print(f"⏭️  Skipping {image_file} (already processed)")
            skipped_count += 1
            continue
    except:
        pass  # Image not in collection, proceed
    
    print(f"\n{'='*70}")
    print(f"Processing {idx}/{len(image_files)}: {image_file}")
    print(f"{'='*70}")
    
    try:
        # Generate description with GPT-4o
        image_path = os.path.join(current_dir, image_file)
        print(f"📷 Analyzing with GPT-4o vision...")
        
        description = describe_image(image_path)
        
        print(f"✅ Description generated ({len(description)} chars)")
        print(f"Preview: {description[:150]}...\n")
        
        # Extract model info from filename if possible
        equipment_type = "Unknown"
        model = "Unknown"
        
        if "radio" in image_file.lower():
            equipment_type = "Radio Equipment"
        elif "air" in image_file.lower():
            equipment_type = "AIR Equipment"
        elif "ssr" in image_file.lower():
            equipment_type = "SSR Equipment"
        
        # Store in ChromaDB
        collection.add(
            documents=[description],
            metadatas=[{
                "image_filename": image_file,
                "equipment_type": equipment_type,
                "model": model,
                "description_method": "GPT-4o Vision",
                "processed_at": time.strftime("%Y-%m-%d %H:%M:%S")
            }],
            ids=[image_id]
        )
        
        print(f"💾 Stored in ChromaDB with ID: {image_id}")
        processed_count += 1
        
        # Small delay to avoid rate limits
        time.sleep(1)
        
    except Exception as e:
        print(f"❌ Error processing {image_file}: {e}")

# Final summary
print(f"\n{'='*70}")
print(f"✅ Batch Processing Complete!")
print(f"{'='*70}")
print(f"📊 Images processed: {processed_count}")
print(f"⏭️  Images skipped: {skipped_count}")
print(f"📚 Total documents in collection: {collection.count()}")

In [ ]:
# ========================================================================
# Cell 11: Comprehensive Image RAG Test - Search All Equipment
# ========================================================================
# Purpose: Demonstrate Image RAG across all 6 Ericsson equipment images
# ========================================================================

print("🚀 COMPREHENSIVE IMAGE RAG DEMO")
print(f"📚 Searching across {collection.count()} equipment images\n")

# Test 1: General equipment search
print("\n" + "="*70)
print("TEST 1: What Ericsson equipment do we have?")
print("="*70)
result1 = search_images("Show me all Ericsson telecom equipment", n_results=6)

# Test 2: Search for specific features
print("\n" + "="*70)
print("TEST 2: Equipment with network ports")
print("="*70)
result2 = search_images("Equipment with Ethernet or network ports", n_results=3)

# Test 3: Search by installation type
print("\n" + "="*70)
print("TEST 3: Rooftop or outdoor equipment")
print("="*70)
result3 = search_images("Rooftop or outdoor telecom equipment", n_results=3)

# Test 4: Search by equipment racks
print("\n" + "="*70)
print("TEST 4: Equipment racks or systems")
print("="*70)
result4 = search_images("Telecom equipment racks or server systems", n_results=3)

# Summary
print("\n" + "="*70)
print("✅ IMAGE RAG DEMO COMPLETE!")
print("="*70)
print(f"📊 Total images indexed: {collection.count()}")
print(f"🔍 Queries tested: 4")
print(f"💡 Key Achievement: Semantic image search working perfectly!")
print(f"\n🎯 Use Case for Ericsson:")
print(f"   - Field technicians can search equipment by description")
print(f"   - Find equipment by features (ports, mounting type, etc.)")
print(f"   - No need for manual tagging - AI generates searchable descriptions")

In [6]:
%%writefile app.py
# ========================================================================
# ERICSSON EQUIPMENT SEARCH - WEB USER INTERFACE
# ========================================================================
# Purpose: Provide a ChatGPT-like web interface for equipment search
#
# Technology Stack:
# - Streamlit: Python web framework (creates UI automatically)
# - ChromaDB: Vector database (loaded from saved .pkl file)
# - Azure OpenAI: For query embeddings only (images already processed)
#
# User Flow:
# 1. User opens web page → sees search interface
# 2. User clicks "Load Database" → system loads pre-processed images
# 3. User types query → system searches and displays results
# ========================================================================

import streamlit as st        # Web UI framework
import requests              # For API calls to Azure OpenAI
import base64                # Not used here but imported for consistency
import chromadb              # Vector database
import pickle                # For loading saved image data
import os                    # For file operations

# ========================================================================
# PAGE CONFIGURATION
# ========================================================================
# What: Sets up the web page appearance and metadata
# Why: Creates professional look and browser tab customization
# ========================================================================

st.set_page_config(
    page_title="Ericsson Equipment Search",  # Browser tab title
    page_icon="📡",                          # Browser tab icon
    layout="wide"                            # Use full width (not centered)
)

# ========================================================================
# AZURE OPENAI CONFIGURATION
# ========================================================================
# What: Credentials for Azure OpenAI API
# Why: Need to convert user queries to embeddings for vector search
#
# Note: We DON'T use this for images (already processed and saved)
#       We ONLY use this for converting user search queries to vectors
# ========================================================================

API_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"

# ========================================================================
# PAGE HEADER
# ========================================================================
# What: Title and description visible to users
# Why: Explains what the system does
# ========================================================================

st.title("📡 Ericsson Equipment Search")
st.markdown("**AI-powered semantic search across telecom equipment images**")
st.markdown("Search using natural language - find equipment by features, not just keywords")
st.markdown("---")  # Horizontal line separator

# ========================================================================
# HELPER FUNCTION: Get Embedding from Text
# ========================================================================
# What: Converts user's search query to embedding vector
# Why: Need vector representation to search ChromaDB
#
# Process:
# 1. User types: "radio equipment"
# 2. This function converts it to: [0.123, -0.456, ... 1536 numbers]
# 3. ChromaDB compares this vector against stored image vectors
# 4. Returns closest matches
#
# Input: Text string (user's search query)
# Output: List of 1536 numbers (embedding vector)
# ========================================================================

def get_embedding(text):
    """
    Convert search query to embedding vector using Ada-002
    
    Args:
        text (str): User's search query (e.g., "ethernet ports")
    
    Returns:
        list: 1536-dimensional embedding vector
    """
    # Construct API URL for embeddings endpoint
    url = f"{ENDPOINT}openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01"
    
    # Set authentication headers
    headers = {
        "api-key": API_KEY,
        "Content-Type": "application/json"
    }
    
    # Make API request
    response = requests.post(url, headers=headers, json={"input": text})
    
    # Extract and return embedding vector
    return response.json()["data"][0]["embedding"]

# ========================================================================
# SESSION STATE MANAGEMENT
# ========================================================================
# What: Streamlit's way of maintaining state across user interactions
# Why: Need to remember if database is loaded (avoids reloading on every click)
#
# Session State Variables:
# - initialized: Boolean, True if database is loaded
# - collection: ChromaDB collection object (the vector database)
#
# How it works:
# - First visit: initialized = False, show "Load Database" button
# - After loading: initialized = True, show search interface
# - On page refresh: state persists, no need to reload
# ========================================================================

if 'initialized' not in st.session_state:
    st.session_state.initialized = False  # Default: not loaded yet

# ========================================================================
# DATABASE LOADING SECTION
# ========================================================================
# What: Initial state - show button to load equipment database
# Why: Don't auto-load (user controls when to initialize)
#
# This section only displays when system is NOT initialized
# ========================================================================

if not st.session_state.initialized:
    # Display load button
    # type="primary" makes it blue and prominent
    if st.button("🚀 Load Equipment Database", type="primary"):
        
        # Show loading spinner while processing
        # Why: User feedback during the 1-2 second load time
        with st.spinner("Loading equipment database..."):
            
            # ================================================================
            # STEP 1: Load Pre-Processed Data from Disk
            # ================================================================
            # What: Read the .pkl file created during setup
            # Contains: All 5 images with descriptions and embeddings
            #
            # File structure:
            # [
            #   {
            #     "filename": "radio-dot-8413-high.jpg",
            #     "description": "This is an Ericsson...",
            #     "embedding": [0.123, -0.456, ... 1536 numbers]
            #   },
            #   ... 4 more images
            # ]
            # ================================================================
            
            with open("ericsson_image_data.pkl", "rb") as f:
                saved_data = pickle.load(f)
            
            # ================================================================
            # STEP 2: Initialize ChromaDB Vector Database
            # ================================================================
            # What: Create in-memory vector database
            # Why: Enables fast similarity search
            # ================================================================
            
            chroma = chromadb.Client()  # Create ChromaDB client
            
            # Delete old collection if exists (ensures fresh start)
            try:
                chroma.delete_collection("demo")
            except:
                pass  # Collection doesn't exist, that's fine
            
            # Create new collection
            col = chroma.create_collection("demo")
            
            # ================================================================
            # STEP 3: Load Saved Data into ChromaDB
            # ================================================================
            # What: Populate database with pre-processed embeddings
            # Why: Makes images searchable without reprocessing
            #
            # This is FAST because:
            # - No GPT-4o Vision calls (already done)
            # - No Ada-002 embedding calls (already done)
            # - Just loading data into memory
            # ================================================================
            
            for i, item in enumerate(saved_data):
                col.add(
                    embeddings=[item["embedding"]],       # Pre-computed 1536-D vector
                    documents=[item["description"]],      # Pre-generated description
                    metadatas=[{"filename": item["filename"]}],  # Image filename
                    ids=[f"img_{i}"]                      # Unique ID
                )
            
            # ================================================================
            # STEP 4: Save to Session State
            # ================================================================
            # What: Store collection in session state
            # Why: Keeps database loaded across button clicks/searches
            # ================================================================
            
            st.session_state.collection = col
            st.session_state.initialized = True
            
            # Show success message
            st.success(f"✅ Loaded {len(saved_data)} equipment images!")
            
            # Rerun the app to show search interface
            # Why: Streamlit needs to re-render to show new state
            st.rerun()

# ========================================================================
# SEARCH INTERFACE SECTION
# ========================================================================
# What: Main search interface (shown after database is loaded)
# Why: This is what users interact with to search equipment
#
# This section only displays when system IS initialized
# ========================================================================

else:
    # ====================================================================
    # Status Indicator
    # ====================================================================
    # What: Shows system is ready to search
    # Why: User confirmation that system is operational
    # ====================================================================
    
    st.success("✅ System Ready - Database Loaded")
    
    # ====================================================================
    # Search Input Field
    # ====================================================================
    # What: Text box for user to enter search query
    # Why: Primary user input for the search system
    #
    # Features:
    # - Placeholder text shows example queries
    # - User can type any natural language query
    # ====================================================================
    
    query = st.text_input(
        "🔍 Search equipment:",
        placeholder="e.g., radio equipment with ethernet ports"
    )
    
    # ====================================================================
    # Results Slider
    # ====================================================================
    # What: Let user control how many results to show
    # Why: Flexibility - sometimes want 1 result, sometimes want 5
    #
    # Parameters:
    # - Minimum: 1 result
    # - Maximum: 5 results
    # - Default: 3 results
    # ====================================================================
    
    num_results = st.slider("Number of results to show:", 1, 5, 3)
    
    # ====================================================================
    # Search Button and Results Display
    # ====================================================================
    # What: Executes search when clicked
    # Why: User controls when to search (not on every keystroke)
    #
    # Condition: Only execute if button clicked AND query is not empty
    # ====================================================================
    
    if st.button("Search", type="primary") and query:
        
        # Show spinner during search
        # Why: Search takes 1-2 seconds (API call + vector search)
        with st.spinner("Searching equipment database..."):
            
            # ============================================================
            # SEARCH PROCESS
            # ============================================================
            # Step 1: Convert query to embedding vector
            # Why: Need vector to compare against stored image vectors
            # ============================================================
            
            query_embedding = get_embedding(query)
            
            # ============================================================
            # Step 2: Search ChromaDB for similar vectors
            # Why: Find images whose descriptions are semantically similar
            #
            # How it works:
            # - ChromaDB compares query vector against all stored vectors
            # - Uses cosine similarity (mathematical measure of similarity)
            # - Returns top N closest matches
            # ============================================================
            
            results = st.session_state.collection.query(
                query_embeddings=[query_embedding],  # Your search query vector
                n_results=num_results                # How many results to return
            )
            
            # ============================================================
            # RESULTS DISPLAY
            # ============================================================
            # What: Show search results in a user-friendly format
            # Why: Users need to see which images match their query
            # ============================================================
            
            st.markdown("---")  # Separator line
            st.markdown(f"### 📊 Results for: *{query}*")
            
            # Loop through each result
            # zip combines documents (descriptions) and metadatas (filenames)
            for i, (doc, meta) in enumerate(zip(
                results['documents'][0],   # List of matching descriptions
                results['metadatas'][0]    # List of corresponding filenames
            )):
                # ========================================================
                # Two-Column Layout for Each Result
                # ========================================================
                # What: Split display into image (left) and description (right)
                # Why: Visual + text provides best user experience
                #
                # Column widths: [1, 3]
                # - Column 1: 25% width (image)
                # - Column 2: 75% width (description)
                # ========================================================
                
                col1, col2 = st.columns([1, 3])
                
                # Left column: Display image
                with col1:
                    img_path = meta['filename']
                    
                    # Check if image file exists in current directory
                    if os.path.exists(img_path):
                        st.image(
                            img_path,
                            use_container_width=True  # Scale to fit column
                        )
                    else:
                        # Image not found (might be in different directory)
                        st.info("📷 Image not available")
                
                # Right column: Display filename and description
                with col2:
                    st.markdown(f"**Result {i+1}: {meta['filename']}**")
                    st.write(doc)  # The AI-generated description
                
                st.markdown("---")  # Separator between results

# ========================================================================
# END OF APPLICATION
# ========================================================================

Writing app.py


In [7]:
!ls -lh app.py requirements.txt ericsson_image_data.pkl

ls: cannot access 'requirements.txt': No such file or directory
ls: cannot access 'ericsson_image_data.pkl': No such file or directory
-rwxrwxrwx 1 root root 16K Dec 14 04:43 app.py


In [1]:
%%writefile requirements.txt
streamlit
requests
chromadb==0.4.22

Overwriting requirements.txt


In [3]:
!ls -lh *.pkl

ls: cannot access '*.pkl': No such file or directory


In [4]:
# ========================================================================
# CREATE DATA FILE - Process images and save to disk
# ========================================================================

import requests, base64, pickle, os

API_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"

def describe_image(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    url = f"{ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01"
    headers = {"api-key": API_KEY, "Content-Type": "application/json"}
    data = {"messages": [{"role": "user", "content": [
        {"type": "text", "text": "Describe this Ericsson telecom equipment in detail."},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}
    ]}], "max_tokens": 200}
    return requests.post(url, headers=headers, json=data).json()["choices"][0]["message"]["content"]

def get_embedding(text):
    url = f"{ENDPOINT}openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01"
    headers = {"api-key": API_KEY, "Content-Type": "application/json"}
    return requests.post(url, headers=headers, json={"input": text}).json()["data"][0]["embedding"]

print("🚀 Processing images...")
imgs = [f for f in os.listdir('.') if f.endswith('.jpg')]
data = []

for i, img in enumerate(imgs):
    print(f"[{i+1}/{len(imgs)}] {img}")
    desc = describe_image(img)
    emb = get_embedding(desc)
    data.append({"filename": img, "description": desc, "embedding": emb})

with open("ericsson_image_data.pkl", "wb") as f:
    pickle.dump(data, f)

print(f"\n✅ Saved: ericsson_image_data.pkl ({len(data)} images)")

🚀 Processing images...
[1/5] AIR_6428_Rooftop_16bit_sRGB_148412 (2).jpg
[2/5] ericsson-radio-system-family-mounting-alternatives-59625-low.jpg
[3/5] radio-dot-8413-high.jpg
[4/5] radios_5g_ready_02.jpg
[5/5] ssr8010_front3.jpg

✅ Saved: ericsson_image_data.pkl (5 images)


In [5]:
!ls -lh app.py requirements.txt ericsson_image_data.pkl *.jpg

-rwxrwxrwx 1 root root 202K Dec 13 09:26 'AIR_6428_Rooftop_16bit_sRGB_148412 (2).jpg'
-rwxrwxrwx 1 root root  16K Dec 14 04:43  app.py
-rwxrwxrwx 1 root root  36K Dec 13 09:25  ericsson-radio-system-family-mounting-alternatives-59625-low.jpg
-rwxrwxrwx 1 root root  74K Dec 14 07:11  ericsson_image_data.pkl
-rwxrwxrwx 1 root root 147K Dec 13 09:26  radio-dot-8413-high.jpg
-rwxrwxrwx 1 root root  34K Dec 13 09:26  radios_5g_ready_02.jpg
-rwxrwxrwx 1 root root   36 Dec 14 06:59  requirements.txt
-rwxrwxrwx 1 root root 846K Dec 13 09:25  ssr8010_front3.jpg
